# 10. Reviewing a Finished Stack

In this tutorial, you will learn how to check a stack after it is made: make its preview picture again without stacking, see why frames were set aside, and compare the stack with the one before it.


## 10.1 Making the Preview Picture Again

The preview picture is a stretched copy of the stack that people look at. Changing a picture setting does not change the stack, so `astrometrics.processing.remake_preview()` runs only the picture step. The choices you pass apply to this one run and are not saved to the settings.

The result carries `preview_settings`, read from a small record saved beside the stack. It says what really made the picture. If the picture has no record, `preview_settings` is `None`, which means unknown. Do not read `None` as the app default.


In [1]:
from astrometricslib import Astrometrics

astrometrics = Astrometrics()

result = astrometrics.processing.remake_preview("M 13", denoise=True, denoise_strength=0.5, star_toning=False)
print("New picture:", result.preview_path)
print("Shown in the viewer:", result.shown_in_viewer)
if result.preview_settings is None:
    print("What made this picture is unknown: no record was saved with it.")
else:
    used = result.preview_settings.settings_used
    print("Made at:", result.preview_settings.made_at)
    print("Noise removal:", used.denoise, "strength", used.denoise_strength)
    print("Star toning:", used.star_toning)
    print("Choices for this run only:", result.preview_settings.custom)

New picture: /home/user/astrometrics/library/stacks/lights/M 13/M_13_L_Stacked_ZWO_ASI_533MM_Pro_405mm_preview.jpg
Shown in the viewer: True
Made at: 2026-11-01T00:58:18+00:00
Noise removal: True strength 0.5
Star toning: False
Choices for this run only: True


`shown_in_viewer` is `False` when you attached your own picture to the target. The new picture was still made, but the viewer keeps showing yours.


## 10.2 Why Frames Were Set Aside

Before a stack, the stacker moves frames with clouds or smeared stars into an `_excluded` folder. `astrometrics.quality.set_aside_frames()` lists them and says why. It only reads: nothing is moved.

The stored kind `trailed` covers two cases: stars smeared, or far fewer stars than usual under a normal sky. The `reason` text says which one it was. A frame with no kind and no reason has no record of why it was set aside.


In [2]:
frames = astrometrics.quality.set_aside_frames("M 13")
print(f"{len(frames)} frames are set aside")
for frame in frames:
    if frame.kind is None and not frame.reason:
        why = "no record of why"
    else:
        why = f"{frame.kind}: {frame.reason}"
    print(f"{frame.file} ({frame.exposure_seconds} s): {why}")
    if not frame.restorable:
        print("   cannot be put back automatically:", frame.restore_note)

0 frames are set aside


## 10.3 Comparing with the Previous Stack

A restack keeps the stack it replaced. `astrometrics.quality.measure_stack()` with `compare_to="previous"` measures both stacks the same way. The noise is the number the target report stores (`stacking.empty_aperture_noise_normalized`), measured in the same empty patches of sky. If a stack has no sky coordinates, the library plate solves it first, which can take about ten minutes when the solve fails.

The result gives one sentence for each number. A number that could not be measured says so and gives the reason. The comparison does not say which stack is better, because that depends on what changed.


In [3]:
report = astrometrics.quality.measure_stack("M 13", compare_to="previous")
if report.comparison is None:
    print(report.notes)
else:
    for sentence in report.comparison.summary:
        print(sentence)
    for note in report.comparison.before.notes + report.comparison.after.notes:
        print("note:", note)

The current archive release 3.15 contains two breaking changes. Please see https://www.cosmos.esa.int/web/gaia/news#GACSrelease315 for details. In case of questions or feedback, please consult the Gaia helpdesk at https://www.cosmos.esa.int/web/gaia/gaia-helpdesk.


INFO: Query finished. [astroquery.utils.tap.core]


sky level 0.00483 to 0.00483 (unchanged).
noise in empty sky 0.0204% of the reference stars' brightness to 0.0204% of the reference stars' brightness (unchanged within the measurement error).
large-scale flatness (rms) 0.6% of the sky to 0.6% of the sky (unchanged).
large-scale flatness (peak to peak) 4.15% of the sky to 4.15% of the sky (unchanged).
star width (FWHM) 2.45 px to 2.45 px (unchanged).
fine-grain noise (pixel to pixel, not the noise number of the target report) 1.07% of the sky to 1.07% of the sky (unchanged).
